# Лабораторная работа № 5 - Сверточные нейронные сети (CNN)

В данной лабораторной работе мы реализуем сверточную нейронную сеть (CNN) для задачи классификации изображений, обучим модель и оптимизируем её для достижения точности не менее 75%.

Задачи:
1. Реализовать сверточную нейронную сеть
2. Обучить модель
3. Добиться точности модели не менее 75%

In [ ]:
# Импорт необходимых библиотек
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader
from torchvision.datasets import CIFAR10
from torchvision.utils import make_grid
from torch.optim.lr_scheduler import StepLR

# Проверка доступности GPU
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(f"Используемое устройство: {device}")

In [ ]:
# Параметры обучения
batch_size = 64
num_epochs = 30
learning_rate = 0.001

# Подготовка данных
transform_train = transforms.Compose([
    transforms.RandomHorizontalFlip(),
    transforms.RandomCrop(32, padding=4),
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])

transform_test = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])

# Загрузка обучающего и тестового набора данных CIFAR10
train_dataset = CIFAR10(root='./data', train=True, download=True, transform=transform_train)
test_dataset = CIFAR10(root='./data', train=False, download=True, transform=transform_test)

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=2)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers=2)

# Классы в датасете CIFAR10
classes = ('самолет', 'автомобиль', 'птица', 'кошка', 'олень', 
           'собака', 'лягушка', 'лошадь', 'корабль', 'грузовик')

In [ ]:
# Визуализация примеров изображений
def imshow(img):
    img = img / 2 + 0.5  # денормализация
    npimg = img.numpy()
    plt.figure(figsize=(10, 4))
    plt.imshow(np.transpose(npimg, (1, 2, 0)))
    plt.axis('off')
    plt.show()

# Получаем случайные изображения из обучающего набора
dataiter = iter(train_loader)
images, labels = next(dataiter)

# Показываем изображения
imshow(make_grid(images[:8]))
# Выводим метки классов
print(' '.join(f'{classes[labels[j]]:5s}' for j in range(8)))

In [ ]:
# Определение архитектуры сверточной нейронной сети
class ConvNet(nn.Module):
    def __init__(self):
        super(ConvNet, self).__init__()
        # Первый сверточный блок
        self.conv1 = nn.Conv2d(in_channels=3, out_channels=32, kernel_size=3, padding=1)
        self.bn1 = nn.BatchNorm2d(32)
        
        # Второй сверточный блок
        self.conv2 = nn.Conv2d(in_channels=32, out_channels=64, kernel_size=3, stride=1, padding=1)
        self.bn2 = nn.BatchNorm2d(64)
        
        # Третий сверточный блок
        self.conv3 = nn.Conv2d(in_channels=64, out_channels=128, kernel_size=3, stride=1, padding=1)
        self.bn3 = nn.BatchNorm2d(128)
        
        # Полносвязные слои для классификации
        self.fc1 = nn.Linear(128 * 4 * 4, 512)
        self.dropout = nn.Dropout(0.5)
        self.fc2 = nn.Linear(512, 10)
        
        # Функции активации и пулинг
        self.relu = nn.ReLU()
        self.pool = nn.MaxPool2d(kernel_size=2, stride=2)

    def forward(self, x):
        # Первый блок
        x = self.relu(self.bn1(self.conv1(x)))
        x = self.pool(x)
        
        # Второй блок
        x = self.relu(self.bn2(self.conv2(x)))
        x = self.pool(x)
        
        # Третий блок
        x = self.relu(self.bn3(self.conv3(x)))
        x = self.pool(x)
        
        # Преобразование для полносвязных слоев
        x = x.view(-1, 128 * 4 * 4)
        
        # Полносвязные слои
        x = self.relu(self.fc1(x))
        x = self.dropout(x)
        x = self.fc2(x)
        
        return x

# Инициализация модели
model = ConvNet().to(device)
print(model)

In [ ]:
# Функция потерь и оптимизатор
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=learning_rate)

# Планировщик скорости обучения
scheduler = StepLR(optimizer, step_size=10, gamma=0.5)

In [ ]:
# Функция для обучения модели
def train_model(model, train_loader, criterion, optimizer, scheduler, num_epochs, device):
    model.train()
    losses = []
    train_acc = []
    
    for epoch in range(num_epochs):
        running_loss = 0.0
        correct = 0
        total = 0
        
        for i, (images, labels) in enumerate(train_loader):
            # Перенос данных на устройство
            images = images.to(device)
            labels = labels.to(device)
            
            # Обнуление градиентов
            optimizer.zero_grad()
            
            # Прямой проход
            outputs = model(images)
            
            # Вычисление функции потерь
            loss = criterion(outputs, labels)
            
            # Обратное распространение ошибки
            loss.backward()
            
            # Оптимизация весов
            optimizer.step()
            
            running_loss += loss.item()
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
            
            if (i + 1) % 100 == 0:
                print(f"Эпоха [{epoch + 1}/{num_epochs}], Шаг [{i + 1}/{len(train_loader)}], "
                      f"Потери: {running_loss / 100:.4f}, "
                      f"Точность: {correct / total:.4f}")
                running_loss = 0.0
        
        epoch_acc = correct / total
        train_acc.append(epoch_acc)
        
        scheduler.step()
        
        epoch_loss = running_loss / len(train_loader)
        losses.append(epoch_loss)
        
        print(f"Эпоха [{epoch + 1}/{num_epochs}], Потери: {epoch_loss:.4f}, "
              f"Точность обучения: {epoch_acc:.4f}, "
              f"Скорость обучения: {scheduler.get_last_lr()[0]:.6f}")
    
    return losses, train_acc

In [ ]:
def evaluate_model(model, test_loader, device):
    model.eval()
    correct = 0
    total = 0
    class_correct = list(0. for i in range(10))
    class_total = list(0. for i in range(10))
    
    with torch.no_grad(): 
        for images, labels in test_loader:
            images = images.to(device)
            labels = labels.to(device)
            
            outputs = model(images)
            _, predicted = torch.max(outputs.data, 1)
            
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
            
            c = (predicted == labels).squeeze()
            for i in range(labels.size(0)):
                label = labels[i]
                class_correct[label] += c[i].item()
                class_total[label] += 1
    
    accuracy = correct / total
    print(f"Точность на тестовом наборе: {accuracy:.4f}")
    
    for i in range(10):
        class_accuracy = class_correct[i] / class_total[i]
        print(f"Точность класса '{classes[i]}': {class_accuracy:.4f}")
    
    return accuracy, class_correct, class_total

In [ ]:
# Обучение модели
losses, train_acc = train_model(model, train_loader, criterion, optimizer, scheduler, num_epochs, device)

# Оценка модели
print("\nОценка модели на тестовом наборе:")
accuracy, class_correct, class_total = evaluate_model(model, test_loader, device)

In [ ]:
# Визуализация процесса обучения
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.plot(range(1, num_epochs + 1), train_acc, 'b-', label='Точность обучения')
plt.title('Точность модели в процессе обучения')
plt.xlabel('Эпоха')
plt.ylabel('Точность')
plt.legend()
plt.grid(True)

plt.subplot(1, 2, 2)
plt.plot(range(1, num_epochs + 1), losses, 'r-', label='Функция потерь')
plt.title('Функция потерь в процессе обучения')
plt.xlabel('Эпоха')
plt.ylabel('Потери')
plt.legend()
plt.grid(True)

plt.tight_layout()
plt.show()

In [ ]:
# Визуализация точности по классам
class_accuracy = [class_correct[i] / class_total[i] for i in range(10)]

plt.figure(figsize=(12, 6))
plt.bar(classes, class_accuracy, color='skyblue')
plt.title('Точность модели по классам')
plt.xlabel('Класс')
plt.ylabel('Точность')
plt.ylim([0, 1])
plt.xticks(rotation=45)
plt.grid(axis='y')

# Добавим значения точности над каждым столбцом
for i, v in enumerate(class_accuracy):
    plt.text(i, v + 0.02, f"{v:.2f}", ha='center')

plt.tight_layout()
plt.show()

In [ ]:
# Сохранение обученной модели
torch.save(model.state_dict(), 'cnn_cifar10_model.pth')
print("Модель успешно сохранена в файл 'cnn_cifar10_model.pth'")

# Проверка общей точности с требованием в 75%
if accuracy >= 0.75:
    print(f"Задание выполнено успешно! Достигнутая точность: {accuracy:.4f} (требуемая: 0.75)")
else:
    print(f"Модель не достигла требуемой точности. Текущая точность: {accuracy:.4f} (требуемая: 0.75)")